In [1]:
import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    "openvla/openvla-7b",
    trust_remote_code=True,
)
print("Processor type:", type(processor).__name__)

vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)

print("Model 로드 완료")
print(f"GPU 메모리 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

image = Image.fromarray(
    (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
)
instruction = "pick up the can"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
print(f"inputs.keys(): {list(inputs.keys())}")

# attention_mask 는 전달하지 않는다 -- predict_action 이 빈 토큰(29871) 을 input_ids 에만
# 덧붙여 mask 와 길이가 1 어긋나므로 (eager attention 에서 크래시), generate 가 mask 를
# 알아서 생성하게 둔다
with torch.no_grad():
    action = vla.predict_action(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        unnorm_key="bridge_orig",
        do_sample=False,
    )
print(f"Action shape: {action.shape}")
print(f"Action : {action}")
print(f"GPU 메모리 : {torch.cuda.memory_allocated()/1e9:.2f} GB")


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processor type: PrismaticProcessor


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.55s/it]
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model 로드 완료
GPU 메모리 사용: 4.38 GB
inputs.keys(): ['input_ids', 'attention_mask', 'pixel_values']
Action shape: (7,)
Action : [-2.08787322e-04  5.40355426e-03 -2.12870912e-02  5.60846725e-03
 -1.41778920e-02  7.89113943e-02  9.96078431e-01]
GPU 메모리 : 4.39 GB


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
import time
import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)

image = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype(np.uint8))
instruction = "pick up the can"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"
inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)

for i in range(5):
    with torch.no_grad():
        action = vla.predict_action(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            unnorm_key="bridge_orig", do_sample=False,
        )
print("warm-up 완료")

latencies = []

for i in range(100):
    # 새 이미지 (cache 효과 방지) -- warm-up 과 동일한 3채널 RGB
    image = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype(np.uint8))
    inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)

    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        action = vla.predict_action(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            unnorm_key="bridge_orig", do_sample=False,
        )
    torch.cuda.synchronize()
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)

    if (i + 1) % 10 == 0:
        print(f"{i+1}/100: latest = {elapsed_ms:.1f} ms")

arr = np.array(latencies)
print("\n[2-4] Latency 통계")
print(f"mean : {arr.mean():.1f} ms")
print(f"median : {np.median(arr):.1f} ms")
print(f"std : {arr.std():.1f} ms")
print(f"min : {arr.min():.1f} ms")
print(f"max : {arr.max():.1f} ms")
print(f"p95 : {np.percentile(arr, 95):.1f} ms")
print(f"p99 : {np.percentile(arr, 99):.1f} ms")
print()
print(f"Throughput : {1000 / arr.mean():.2f} Hz")

np.save("openvla_latency_4070_int4.npy", arr)
print("\n 결과 저장: openvla_latency_4070_int4.npy")
print("-> Phase 7 의 산출물 v3 에서 비교 baseline 으로 사용")